# Pipeline Accuracy & Scale Tests

> Direct pipeline invocation — no HTTP server needed.  
> Full LangGraph pipeline: **supervisor → math_agent → reviewer → formatter → END**

| # | Test | Topic | Items | Game Types | Kiểm tra |
|---|------|-------|-------|------------|----------|
| 1 | Accuracy 10Q | Đạo hàm và ứng dụng | 10 | quiz | Computation traces, 4 options, correct answer |
| 2 | Vietnamese | Phương trình bậc hai | 10 | quiz + flashcard | All content in Vietnamese |
| 3 | Diversity 20Q | Hàm số và đồ thị | 20 | quiz | Unique questions, sub-topic distribution |
| 4 | Scale 30Q | Tích phân và ứng dụng | 30 | quiz | Handles Kahoot-scale, time per item |
| 5 | All Game Types | Chuyển động thẳng đều | 20 | quiz+flashcard+fill_blank | All 3 formats work |

## Setup & Imports

In [1]:
import sys, time, re, json
from pathlib import Path
from IPython.display import display, Markdown, HTML

project_root = str(Path.cwd().parent.parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config import get_settings
from src.api.schemas import (
    GenerationRequest, GameType, DifficultyLevel, DocScope,
    GameContentResponse, QuizQuestion, Flashcard, FillBlankQuestion,
)
from src.graph.builder import compile_graph
from src.graph.state import MAX_REVIEW_ITERATIONS

settings = get_settings()

display(Markdown(f"""
### Configuration
| Setting | Value |
|---------|-------|
| GCP Project | `{settings.gcp_project_id}` |
| Generation Model | `{settings.generation_model}` @ `{settings.generation_model_location or settings.gcp_location}` |
| Review Model | `{settings.review_model}` @ `{settings.review_model_location}` |
| Max Review Iterations | `{MAX_REVIEW_ITERATIONS}` |
"""))
print('Setup OK ✓')

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



### Configuration
| Setting | Value |
|---------|-------|
| GCP Project | `green-mercury-485016-n1` |
| Generation Model | `gemini-2.5-flash` @ `asia-southeast1` |
| Review Model | `gemini-3.1-flash-lite-preview` @ `global` |
| Max Review Iterations | `3` |


Setup OK ✓


## Helper Functions

In [2]:
# ── Request builder ──
def make_request(
    topic: str,
    num_questions: int = 10,
    game_types: list[str] | None = None,
    difficulty: str = 'medium',
    doc_scope: str = 'system',
) -> GenerationRequest:
    gts = [GameType(g) for g in (game_types or ['quiz'])]
    return GenerationRequest(
        user_id='accuracy_test',
        topic=topic,
        game_types=gts,
        num_questions=num_questions,
        difficulty=DifficultyLevel(difficulty),
        doc_scope=DocScope(doc_scope),
    )

# ── Pipeline runner ──
async def run_pipeline(request: GenerationRequest) -> dict:
    app = compile_graph()
    initial_state = {
        'request': request,
        'doc_scope': request.doc_scope.value,
        'iteration_count': 0,
        'rejected_items': [],
        'errors': [],
    }
    t0 = time.time()
    result = await app.ainvoke(initial_state)
    elapsed = time.time() - t0
    return {**result, '_elapsed': elapsed}

# ── Vietnamese detection ──
VN_CHARS = re.compile(r'[àáảãạăắằẳẵặâấầẩẫậèéẻẽẹêếềểễệìíỉĩịòóỏõọôốồổỗộơớờởỡợùúủũụưứừửữựỳýỷỹỵđ]', re.IGNORECASE)
def is_vietnamese(text: str) -> bool:
    return bool(VN_CHARS.search(text))

# ── Rich display helpers ──
def show_summary_table(output: GameContentResponse, elapsed: float, iterations: int) -> dict:
    c = output.content
    m = output.metadata
    qn, fn, bn = len(c.quiz), len(c.flashcard), len(c.fill_blank)
    total = qn + fn + bn
    pr = m.total_passed_review / m.total_generated * 100 if m.total_generated > 0 else 0

    display(Markdown(f"""
| Metric | Value |
|--------|-------|
| Total items | **{total}** |
| Quiz / Flashcard / Fill-blank | {qn} / {fn} / {bn} |
| Generated → Passed review | {m.total_generated} → {m.total_passed_review} ({pr:.0f}%) |
| Rejected | {m.total_rejected} |
| Iterations | {iterations} |
| Time | {elapsed:.1f}s ({elapsed/total:.1f}s per item) |
"""))

    return {'quiz': qn, 'fc': fn, 'fb': bn, 'total': total, 'elapsed': elapsed,
            'generated': m.total_generated, 'passed': m.total_passed_review, 'rejected': m.total_rejected}

def show_quiz_card(q: QuizQuestion, idx: int):
    correct_opt = next((o for o in q.options if o.is_correct), None)
    options_md = '\n'.join(f'  - {"✅" if o.is_correct else "❌"} {o.text}' for o in q.options)
    trace_md = ''
    if q.computation_trace:
        trace_preview = q.computation_trace[:300]
        trace_md = f'\n\n<details><summary>🔢 Computation trace</summary>\n\n```\n{trace_preview}\n```\n</details>'

    display(Markdown(f"""
---
**Q{idx+1}** [{q.topic}] — _{q.difficulty}_

> {q.question}

{options_md}

💡 **Explanation:** {q.explanation[:200]}
{trace_md}
"""))

def show_flashcard(fc: Flashcard, idx: int):
    display(Markdown(f"""
---
**FC{idx+1}** [{fc.topic}]

| | |
|---|---|
| **Front** | {fc.front[:200]} |
| **Back** | {fc.back[:200]} |
"""))

def show_fillblank(fb: FillBlankQuestion, idx: int):
    blanks_str = ', '.join(f'`{b.correct_answer}`' for b in fb.blanks)
    display(Markdown(f"""
---
**FB{idx+1}** [{fb.topic}]

> {fb.template[:200]}

Answers: {blanks_str}
"""))

# Store all results
ALL_RESULTS = {}
print('Helpers loaded ✓')

Helpers loaded ✓


---
## Test 1: Accuracy — 10 Quiz Questions

**Topic:** Đạo hàm và ứng dụng  
**Checks:** Each quiz has 4 options, 1 correct, explanation, and a computation trace.

In [3]:
req1 = make_request('Đạo hàm và ứng dụng', num_questions=10, game_types=['quiz'])
res1 = await run_pipeline(req1)

out1 = res1['final_output']
assert out1 is not None, 'No output!'

stats1 = show_summary_table(out1, res1['_elapsed'], res1['iteration_count'])
ALL_RESULTS['accuracy_10q'] = stats1

2026-03-21 12:18:36 [info     ] creating_graph                


2026-03-21 12:18:36 [info     ] graph_created                 


2026-03-21 12:18:36 [info     ] graph_compiled_without_checkpointer


2026-03-21 12:18:36 [info     ] supervisor_analyzing           doc_scope=system topic='Đạo hàm và ứng dụng' user_id=accuracy_test


2026-03-21 12:18:36 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0


...

2026-03-21 12:20:14 [info     ] formatter_starting             game_types=['quiz'] items_count=10 user_id=accuracy_test


2026-03-21 12:20:22 [info     ] formatter_quiz_batch_parsed    batch_start=0 items=7


2026-03-21 12:20:28 [info     ] formatter_quiz_batch_parsed    batch_start=7 items=3


2026-03-21 12:20:28 [info     ] formatter_completed            fill_blank=0 flashcard=0 generation_time=14.415685653686523 quiz=10 total_items=10



| Metric | Value |
|--------|-------|
| Total items | **10** |
| Quiz / Flashcard / Fill-blank | 10 / 0 / 0 |
| Generated → Passed review | 10 → 10 (100%) |
| Rejected | 0 |
| Iterations | 1 |
| Time | 112.7s (11.3s per item) |


In [4]:
# Validate structure & show all questions
quizzes1 = out1.content.quiz
with_trace = [q for q in quizzes1 if q.computation_trace]

display(Markdown(f'### Computation traces: {len(with_trace)}/{len(quizzes1)}'))

for i, q in enumerate(quizzes1):
    assert len(q.options) == 4, f'Q{i+1}: Expected 4 options, got {len(q.options)}'
    assert any(o.is_correct for o in q.options), f'Q{i+1}: No correct option'
    assert q.question and q.explanation, f'Q{i+1}: Missing question or explanation'
    show_quiz_card(q, i)

display(Markdown('### ✅ Accuracy test PASSED'))

### Computation traces: 10/10


---
**Q1** [Quy tắc tính đạo hàm - Đạo hàm của $x^n$.] — _DifficultyLevel.MEDIUM_

> Đạo hàm của hàm số $f(x) = x^5$ là gì?

  - ✅ $f'(x) = 5x^4$
  - ❌ $f'(x) = x^4$
  - ❌ $f'(x) = 5x^5$
  - ❌ $f'(x) = 4x^5$

💡 **Explanation:** Áp dụng quy tắc đạo hàm của hàm số lũy thừa $(x^n)' = n \cdot x^{n-1}$.


<details><summary>🔢 Computation trace</summary>

```
Code:
```python
import sympy
x = sympy.Symbol('x')
f_x = x**5
f_prime_x = sympy.diff(f_x, x)
print(f'{f_prime_x=}')
```
Output:
f_prime_x=5*x**4
```
</details>



---
**Q2** [Quy tắc tính đạo hàm - Quy tắc tổng, Đạo hàm của $x^n$, Đạo hàm của hằng số.] — _DifficultyLevel.MEDIUM_

> Đạo hàm của hàm số $f(x) = 3x^2 + 5x - 7$ là gì?

  - ❌ $f'(x) = 6x + 5x - 7$
  - ❌ $f'(x) = 3x + 5$
  - ✅ $f'(x) = 6x + 5$
  - ❌ $f'(x) = 6x^2 + 5$

💡 **Explanation:** Áp dụng quy tắc đạo hàm của tổng, hiệu và đạo hàm của hàm số lũy thừa.


<details><summary>🔢 Computation trace</summary>

```
Code:
```python
import sympy
x = sympy.Symbol('x')
f_x = 3*x**2 + 5*x - 7
f_prime_x = sympy.diff(f_x, x)
print(f'{f_prime_x=}')
```
Output:
f_prime_x=6*x + 5
```
</details>



---
**Q3** [Đạo hàm tại một điểm.] — _DifficultyLevel.MEDIUM_

> Tính đạo hàm của hàm số $f(x) = x^3 - 2x + 1$ tại điểm $x_0 = 2$.

  - ❌ $f'(2) = 3x^2 - 2$
  - ❌ $f'(2) = 9$
  - ✅ $f'(2) = 10$
  - ❌ $f'(2) = 12$

💡 **Explanation:** Đầu tiên tính đạo hàm $f'(x) = 3x^2 - 2$, sau đó thay $x=2$ vào biểu thức đạo hàm để được $f'(2) = 3(2)^2 - 2 = 10$.


<details><summary>🔢 Computation trace</summary>

```
Code:
```python
import sympy
x = sympy.Symbol('x')
f_x = x**3 - 2*x + 1
f_prime_x = sympy.diff(f_x, x)
result = f_prime_x.subs(x, 2)
print(f'{result=}')
```
Output:
result=10
```
</details>



---
**Q4** [Quy tắc tính đạo hàm - Quy tắc tích.] — _DifficultyLevel.MEDIUM_

> Đạo hàm của hàm số $f(x) = (x^2 + 1)(2x - 3)$ là gì?

  - ❌ $f'(x) = 2x(2x - 3)$
  - ❌ $f'(x) = (x^2 + 1)2$
  - ✅ $f'(x) = 6x^2 - 6x + 2$
  - ❌ $f'(x) = 2x(2x - 3) + (x^2 + 1)2$

💡 **Explanation:** Áp dụng quy tắc đạo hàm của tích $(uv)' = u'v + uv'$. Với $u=x^2+1, v=2x-3$, ta có $u'=2x, v'=2$. Vậy $f'(x) = 2x(2x-3) + (x^2+1)2 = 4x^2-6x+2x^2+2 = 6x^2-6x+2$.


<details><summary>🔢 Computation trace</summary>

```
Code:
```python
import sympy
x = sympy.Symbol('x')
f_x = (x**2 + 1)*(2*x - 3)
f_prime_x = sympy.diff(f_x, x)
print(f'{f_prime_x=}')
```
Output:
f_prime_x=2*x**2 + 2*x*(2*x - 3) + 2
```
</details>


...


---
**Q7** [Quy tắc tính đạo hàm - Quy tắc chuỗi (dạng lũy thừa).] — _DifficultyLevel.MEDIUM_

> Đạo hàm của hàm số $f(x) = (2x + 3)^4$ là gì?

  - ❌ $f'(x) = 4(2x + 3)^3$
  - ❌ $f'(x) = 4(2x + 3)^3 \cdot 2x$
  - ❌ $f'(x) = 8(2x + 3)^4$
  - ✅ $f'(x) = 8(2x + 3)^3$

💡 **Explanation:** Áp dụng quy tắc chuỗi $(u^n)' = n \cdot u^{n-1} \cdot u'$. Với $u=2x+3$, $u'=2$. Vậy $f'(x) = 4(2x+3)^{4-1} • 2 = 8(2x+3)^3$.


<details><summary>🔢 Computation trace</summary>

```
Code:
```python
import sympy
x = sympy.Symbol('x')
f_x = (2*x + 3)**4
f_prime_x = sympy.diff(f_x, x)
print(f'{f_prime_x=}')
```
Output:
f_prime_x=8*(2*x + 3)**3
```
</details>



---
**Q8** [Khái niệm đạo hàm] — _DifficultyLevel.MEDIUM_

> Đạo hàm của hàm số $f(x) = x^2$ là gì?

  - ❌ $f'(x) = x$
  - ✅ $f'(x) = 2x$
  - ❌ $f'(x) = x^2$
  - ❌ $f'(x) = 2x^2$

💡 **Explanation:** Đạo hàm của hàm số $f(x) = x^2$ được tính bằng quy tắc đạo hàm $(x^n)' = nx^{n-1}$, cho kết quả $2x$. Hoặc theo định nghĩa đạo hàm, ta cũng có $f'(x) = 2x$.


<details><summary>🔢 Computation trace</summary>

```
Code:
```python
import sympy
x = sympy.Symbol('x')
f_x = x**2
f_prime_x = sympy.diff(f_x, x)
print(f'{f_prime_x=}')
```
Output:
f_prime_x=2*x
```
</details>



---
**Q9** [Ứng dụng của đạo hàm - Tìm điểm cực trị (tiền đề)] — _DifficultyLevel.MEDIUM_

> Cho hàm số $f(x) = x^3 - 6x^2 + 9x + 1$. Các giá trị của $x$ mà tại đó đạo hàm của hàm số bằng 0 là gì?

  - ❌ $x = 0$ và $x = 1$
  - ❌ $x = -1$ và $x = -3$
  - ✅ $x = 1$ và $x = 3$
  - ❌ $x = 1$

💡 **Explanation:** Đạo hàm của hàm số là $f'(x) = 3x^2 - 12x + 9$. Giải phương trình $3x^2 - 12x + 9 = 0$ ta được $x = 1$ và $x = 3$.


<details><summary>🔢 Computation trace</summary>

```
Code:
```python
import sympy
x = sympy.Symbol('x')
f_x = x**3 - 6*x**2 + 9*x + 1
f_prime_x = sympy.diff(f_x, x)
solutions = sympy.solve(f_prime_x, x)
print(f'{solutions=}')
```
Output:
solutions=[1, 3]
```
</details>



---
**Q10** [Quy tắc tính đạo hàm - Quy tắc tích, quy tắc tổng] — _DifficultyLevel.MEDIUM_

> Đạo hàm của hàm số $f(x) = (x^2 + 2x)(3x - 1)$ là gì?

  - ❌ $f'(x) = 6x^2 + 4x - 2$
  - ❌ $f'(x) = 3x^2 + 6x$
  - ❌ $f'(x) = 9x^2 + 8x - 2$
  - ✅ $f'(x) = 9x^2 + 10x - 2$

💡 **Explanation:** Áp dụng quy tắc đạo hàm của tích $(uv)' = u'v + uv'$, ta có $f'(x) = (2x+2)(3x-1) + (x^2+2x)(3) = (6x^2+4x-2) + (3x^2+6x) = 9x^2 + 10x - 2$.


<details><summary>🔢 Computation trace</summary>

```
Code:
```python
import sympy
x = sympy.Symbol('x')
f_x = (x**2 + 2*x)*(3*x - 1)
f_prime_x = sympy.diff(f_x, x)
print(f'{f_prime_x=}')
```
Output:
f_prime_x=3*x**2 + 6*x + (2*x + 2)*(3*x - 1)
```
</details>


### ✅ Accuracy test PASSED

---
## Test 2: Vietnamese Language Quality

**Topic:** Phương trình bậc hai  
**Checks:** All question text, explanations, and flashcard fronts are in Vietnamese (≥80%).

In [5]:
req2 = make_request('Phương trình bậc hai', num_questions=10, game_types=['quiz', 'flashcard'])
res2 = await run_pipeline(req2)

out2 = res2['final_output']
assert out2 is not None, 'No output!'

stats2 = show_summary_table(out2, res2['_elapsed'], res2['iteration_count'])
ALL_RESULTS['vietnamese'] = stats2

2026-03-21 12:20:29 [info     ] creating_graph                


2026-03-21 12:20:29 [info     ] graph_created                 


2026-03-21 12:20:29 [info     ] graph_compiled_without_checkpointer


2026-03-21 12:20:29 [info     ] supervisor_analyzing           doc_scope=system topic='Phương trình bậc hai' user_id=accuracy_test


2026-03-21 12:20:29 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0


...

2026-03-21 12:23:35 [info     ] formatter_quiz_batch_parsed    batch_start=7 items=3


2026-03-21 12:23:41 [info     ] formatter_flashcard_batch_parsed batch_start=0 items=7


2026-03-21 12:23:48 [info     ] formatter_flashcard_batch_parsed batch_start=7 items=3


2026-03-21 12:23:48 [info     ] formatter_completed            fill_blank=0 flashcard=10 generation_time=33.48262333869934 quiz=10 total_items=20



| Metric | Value |
|--------|-------|
| Total items | **20** |
| Quiz / Flashcard / Fill-blank | 10 / 10 / 0 |
| Generated → Passed review | 16 → 10 (62%) |
| Rejected | 6 |
| Iterations | 2 |
| Time | 199.4s (10.0s per item) |


In [6]:
# Check Vietnamese content
vn_count, total_checked, non_vn = 0, 0, []

for q in out2.content.quiz:
    for label, text in [('Question', q.question), ('Explanation', q.explanation)]:
        total_checked += 1
        if is_vietnamese(text):
            vn_count += 1
        else:
            non_vn.append(f'{label}: {text[:80]}')

for fc in out2.content.flashcard:
    total_checked += 1
    if is_vietnamese(fc.front):
        vn_count += 1
    else:
        non_vn.append(f'FC front: {fc.front[:80]}')

vn_rate = vn_count / total_checked * 100 if total_checked > 0 else 0

display(Markdown(f'### Vietnamese content: {vn_count}/{total_checked} ({vn_rate:.0f}%)'))

if non_vn:
    display(Markdown('**⚠️ Non-Vietnamese items:**\n' + '\n'.join(f'- {x}' for x in non_vn[:5])))

# Show sample quiz & flashcard
display(Markdown('### Sample Quiz'))
if out2.content.quiz:
    show_quiz_card(out2.content.quiz[0], 0)

display(Markdown('### Sample Flashcard'))
if out2.content.flashcard:
    show_flashcard(out2.content.flashcard[0], 0)

assert vn_rate >= 80, f'Expected >=80% Vietnamese, got {vn_rate:.0f}%'
display(Markdown('### ✅ Vietnamese test PASSED'))

### Vietnamese content: 30/30 (100%)

### Sample Quiz


---
**Q1** [Giải phương trình bậc hai cơ bản.] — _DifficultyLevel.MEDIUM_

> Giải phương trình bậc hai sau: $x^2 - 5x + 6 = 0$.

  - ✅ $x_1 = 3$, $x_2 = 2$
  - ❌ $x_1 = -3$, $x_2 = -2$
  - ❌ $x_1 = 3$, $x_2 = -2$
  - ❌ $x_1 = 6$, $x_2 = 1$

💡 **Explanation:** Phương trình đã cho có dạng $ax^2 + bx + c = 0$ với $a=1$, $b=-5$, $c=6$. Bước 1: Tính biệt thức $\Delta$. $\Delta = b^2 - 4ac = (-5)^2 - 4(1)(6) = 25 - 24 = 1$. Bước 2: Vì $\Delta = 1 > 0$, phương tr


<details><summary>🔢 Computation trace</summary>

```
Code:
```python
import sympy

x = sympy.Symbol("x")
roots = sympy.solve(x**2 - 5*x + 6, x)
print(f'{roots=}')
```
Output:
```
roots=[2, 3]
```
```
</details>


### Sample Flashcard


---
**FC1** [Giải phương trình bậc hai cơ bản.]

| | |
|---|---|
| **Front** | Giải phương trình bậc hai sau: $x^2 - 5x + 6 = 0$. |
| **Back** | $x_1 = 3$, $x_2 = 2$. Giải thích: $\Delta = 1 > 0$, phương trình có hai nghiệm phân biệt: $x_1 = \frac{5 + 1}{2} = 3$, $x_2 = \frac{5 - 1}{2} = 2$. |


### ✅ Vietnamese test PASSED

---
## Test 3: Content Diversity — 20 Questions

**Topic:** Hàm số và đồ thị  
**Checks:** All questions are unique (≥90%), multiple sub-topics.

In [7]:
req3 = make_request('Hàm số và đồ thị', num_questions=20, game_types=['quiz'])
res3 = await run_pipeline(req3)

out3 = res3['final_output']
assert out3 is not None, 'No output!'

stats3 = show_summary_table(out3, res3['_elapsed'], res3['iteration_count'])
ALL_RESULTS['diversity_20q'] = stats3

2026-03-21 12:23:48 [info     ] creating_graph                


2026-03-21 12:23:48 [info     ] graph_created                 


2026-03-21 12:23:48 [info     ] graph_compiled_without_checkpointer


2026-03-21 12:23:48 [info     ] supervisor_analyzing           doc_scope=system topic='Hàm số và đồ thị' user_id=accuracy_test


2026-03-21 12:23:48 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0


...

2026-03-21 12:25:54 [info     ] formatter_quiz_batch_parsed    batch_start=0 items=7


2026-03-21 12:26:03 [info     ] formatter_quiz_batch_parsed    batch_start=7 items=7


2026-03-21 12:26:11 [info     ] formatter_quiz_batch_parsed    batch_start=14 items=5


2026-03-21 12:26:11 [info     ] formatter_completed            fill_blank=0 flashcard=0 generation_time=22.326961278915405 quiz=19 total_items=19



| Metric | Value |
|--------|-------|
| Total items | **19** |
| Quiz / Flashcard / Fill-blank | 19 / 0 / 0 |
| Generated → Passed review | 20 → 19 (95%) |
| Rejected | 1 |
| Iterations | 1 |
| Time | 143.3s (7.5s per item) |


In [8]:
quizzes3 = out3.content.quiz
questions3 = [q.question for q in quizzes3]
unique3 = set(questions3)
dup_rate = (len(questions3) - len(unique3)) / len(questions3) * 100 if questions3 else 0

display(Markdown(f'### Unique questions: {len(unique3)}/{len(questions3)} (dup rate: {dup_rate:.0f}%)'))

# Sub-topic distribution
topics3 = {}
for q in quizzes3:
    topics3[q.topic] = topics3.get(q.topic, 0) + 1

topic_rows = '\n'.join(f'| {t} | {c} |' for t, c in sorted(topics3.items(), key=lambda x: -x[1]))
display(Markdown(f"""
### Sub-topic Distribution ({len(topics3)} topics)

| Topic | Count |
|-------|-------|
{topic_rows}
"""))

assert len(unique3) >= len(questions3) * 0.9, f'Too many duplicates: {len(unique3)}/{len(questions3)}'
assert len(topics3) >= 2, f'Expected >=2 sub-topics, got {len(topics3)}'
display(Markdown('### ✅ Diversity test PASSED'))

### Unique questions: 19/19 (dup rate: 0%)


### Sub-topic Distribution (14 topics)

| Topic | Count |
|-------|-------|
| Hàm số chẵn, lẻ | 3 |
| Tập xác định của hàm số | 3 |
| Đạo hàm | 2 |
| Tập xác định của hàm số, Hàm số logarit | 1 |
| Đồ thị hàm số, Giao điểm với trục tọa độ | 1 |
| Sự biến thiên của hàm số, Đạo hàm | 1 |
| Đạo hàm, Tiếp tuyến của đồ thị hàm số | 1 |
| Phương trình đại số | 1 |
| Hàm số mũ | 1 |
| Hàm số logarit, Phương trình logarit | 1 |
| Giá trị cực đại, cực tiểu | 1 |
| Đồ thị hàm số, Đường tiệm cận | 1 |
| Đạo hàm của hàm hợp | 1 |
| Đồ thị hàm số, Giao điểm với trục tọa độ, Phương trình đại số | 1 |


### ✅ Diversity test PASSED

---
## Test 4: Scale Test — 30 Questions (Kahoot Scale)

**Topic:** Tích phân và ứng dụng  
**Checks:** Pipeline handles 30Q (≥70% yield), batch parsing works at scale.

In [9]:
req4 = make_request('Tích phân và ứng dụng', num_questions=30, game_types=['quiz'])
res4 = await run_pipeline(req4)

out4 = res4['final_output']
assert out4 is not None, f'No output. Errors: {res4.get("errors", [])}'

stats4 = show_summary_table(out4, res4['_elapsed'], res4['iteration_count'])
ALL_RESULTS['scale_30q'] = stats4

2026-03-21 12:26:11 [info     ] creating_graph                


2026-03-21 12:26:11 [info     ] graph_created                 


2026-03-21 12:26:11 [info     ] graph_compiled_without_checkpointer


2026-03-21 12:26:11 [info     ] supervisor_analyzing           doc_scope=system topic='Tích phân và ứng dụng' user_id=accuracy_test


2026-03-21 12:26:11 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0


...

2026-03-21 12:29:51 [info     ] formatter_quiz_batch_parsed    batch_start=7 items=7


2026-03-21 12:30:05 [info     ] formatter_quiz_batch_parsed    batch_start=14 items=7


2026-03-21 12:30:16 [info     ] formatter_quiz_batch_parsed    batch_start=21 items=3


2026-03-21 12:30:16 [info     ] formatter_completed            fill_blank=0 flashcard=0 generation_time=45.88103938102722 quiz=24 total_items=24



| Metric | Value |
|--------|-------|
| Total items | **24** |
| Quiz / Flashcard / Fill-blank | 24 / 0 / 0 |
| Generated → Passed review | 24 → 24 (100%) |
| Rejected | 0 |
| Iterations | 1 |
| Time | 244.6s (10.2s per item) |


In [10]:
quizzes4 = out4.content.quiz
n4 = len(quizzes4)
assert n4 >= 21, f'Expected >=21 (70% of 30), got {n4}'

unique4 = set(q.question for q in quizzes4)
display(Markdown(f'### Unique: {len(unique4)}/{n4}'))

# Show first, middle, last
display(Markdown('### Sample Items (first / middle / last)'))
for idx in [0, n4 // 2, n4 - 1]:
    show_quiz_card(quizzes4[idx], idx)

display(Markdown(f'### ✅ Scale test PASSED — {n4} items in {res4["_elapsed"]:.0f}s ({res4["_elapsed"]/n4:.1f}s/item)'))

### Unique: 24/24

### Sample Items (first / middle / last)


---
**Q1** [Tích phân xác định] — _DifficultyLevel.MEDIUM_

> Tính tích phân xác định: $\int_{1}^{2} (x^2 + 2x) dx$

  - ❌ $\frac{20}{3}$
  - ✅ $\frac{16}{3}$
  - ❌ $\frac{4}{3}$
  - ❌ $\frac{10}{3}$

💡 **Explanation:** Để tính tích phân này, ta tìm nguyên hàm của $x^2 + 2x$, là $\frac{x^3}{3} + x^2$. Sau đó, áp dụng Định lý cơ bản của giải tích, ta tính giá trị của nguyên hàm tại cận trên (2) và cận dưới (1) rồi lấy


<details><summary>🔢 Computation trace</summary>

```
Code:
import sympy
from sympy import integrate, Symbol

x = Symbol('x')
f = x**2 + 2*x
result = integrate(f, (x, 1, 2))
print(f'{result=}')

Output:
result=16/3
```
</details>



---
**Q13** [Tích phân xác định bằng phương pháp đổi biến với hàm mũ] — _DifficultyLevel.MEDIUM_

> Tính giá trị của tích phân xác định sau: $\int_{0}^{1} x e^{x^2} \,dx$.

  - ❌ $E - 1$
  - ✅ $\frac{E}{2} - \frac{1}{2}$
  - ❌ $E$
  - ❌ $1/2$

💡 **Explanation:** Sử dụng phép đổi biến $u = x^2$, suy ra $du = 2x \,dx$, hay $x \,dx = \frac{1}{2} \,du$. Khi $x=0$, $u=0$. Khi $x=1$, $u=1$. Tích phân trở thành $\int_{0}^{1} e^u \frac{1}{2} \,du = \frac{1}{2} [e^u]_


<details><summary>🔢 Computation trace</summary>

```
Code:
import sympy
from sympy import integrate, Symbol, exp

x = Symbol('x')
f = x * exp(x**2)
result = integrate(f, (x, 0, 1))
print(f'{result=}')
Output:
result=-1/2 + E/2
```
</details>



---
**Q24** [Tích phân hàm mũ và đa thức] — _DifficultyLevel.MEDIUM_

> Giá trị của tích phân xác định $\int_{0}^{1} (e^x + x) dx$ là bao nhiêu?

  - ✅ $e - \frac{1}{2}$
  - ❌ $e + \frac{1}{2}$
  - ❌ $e + \frac{3}{2}$
  - ❌ $\frac{1}{2}$

💡 **Explanation:** Tích phân của $e^x + x$ là $e^x + \frac{x^2}{2}$. Thay cận từ 0 đến 1, ta được $(e^1 + \frac{1^2}{2}) - (e^0 + \frac{0^2}{2}) = (e + \frac{1}{2}) - (1 + 0) = e - \frac{1}{2}$.


<details><summary>🔢 Computation trace</summary>

```
null
```
</details>


### ✅ Scale test PASSED — 24 items in 245s (10.2s/item)

---
## Test 5: All Game Types — Quiz + Flashcard + Fill-blank

**Topic:** Chuyển động thẳng đều và biến đổi đều (20Q)  
**Checks:** All 3 game formats are generated and structurally valid.

In [11]:
req5 = make_request(
    'Chuyển động thẳng đều và biến đổi đều',
    num_questions=20,
    game_types=['quiz', 'flashcard', 'fill_blank'],
)
res5 = await run_pipeline(req5)

out5 = res5['final_output']
assert out5 is not None, 'No output!'

stats5 = show_summary_table(out5, res5['_elapsed'], res5['iteration_count'])
ALL_RESULTS['all_types'] = stats5

2026-03-21 12:30:16 [info     ] creating_graph                


2026-03-21 12:30:16 [info     ] graph_created                 


2026-03-21 12:30:16 [info     ] graph_compiled_without_checkpointer


2026-03-21 12:30:16 [info     ] supervisor_analyzing           doc_scope=system topic='Chuyển động thẳng đều và biến đổi đều' user_id=accuracy_test


2026-03-21 12:30:16 [debug    ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3.1-flash-lite-preview temperature=0


...

2026-03-21 12:33:50 [info     ] formatter_fill_blank_batch_parsed batch_start=0 items=7


2026-03-21 12:33:55 [info     ] formatter_fill_blank_batch_parsed batch_start=7 items=7


2026-03-21 12:33:59 [info     ] formatter_fill_blank_batch_parsed batch_start=14 items=5


2026-03-21 12:33:59 [info     ] formatter_completed            fill_blank=19 flashcard=19 generation_time=53.16176772117615 quiz=19 total_items=57



| Metric | Value |
|--------|-------|
| Total items | **57** |
| Quiz / Flashcard / Fill-blank | 19 / 19 / 19 |
| Generated → Passed review | 20 → 19 (95%) |
| Rejected | 1 |
| Iterations | 1 |
| Time | 223.3s (3.9s per item) |


In [12]:
qn5, fn5, bn5 = len(out5.content.quiz), len(out5.content.flashcard), len(out5.content.fill_blank)
assert qn5 >= 10, f'Quiz: expected >=10, got {qn5}'
assert fn5 >= 10, f'Flashcard: expected >=10, got {fn5}'
assert bn5 >= 10, f'Fill-blank: expected >=10, got {bn5}'

display(Markdown('### Sample Quiz'))
show_quiz_card(out5.content.quiz[0], 0)

display(Markdown('### Sample Flashcard'))
show_flashcard(out5.content.flashcard[0], 0)

display(Markdown('### Sample Fill-blank'))
show_fillblank(out5.content.fill_blank[0], 0)

display(Markdown('### ✅ All game types test PASSED'))

### Sample Quiz


---
**Q1** [Chuyển động thẳng đều - Quãng đường] — _DifficultyLevel.MEDIUM_

> Một ô tô khởi hành từ A lúc 7 giờ sáng, chuyển động thẳng đều với vận tốc 60 km/h. Hỏi lúc 9 giờ sáng cùng ngày, ô tô cách A bao nhiêu km?

  - ❌ 60 km
  - ❌ 90 km
  - ✅ 120 km
  - ❌ 180 km

💡 **Explanation:** Thời gian ô tô đã đi là: t = 9 giờ - 7 giờ = 2 giờ. Quãng đường ô tô đi được là: s = v * t. Thay số: s = 60 km/h * 2 h. Vậy, lúc 9 giờ sáng, ô tô cách A 120 km.


<details><summary>🔢 Computation trace</summary>

```
Code:
v = 60  # km/h
t = 2   # hours
s = v * t
print(f"Quãng đường s = {s} km")
Output:
Quãng đường s = 120 km
```
</details>


### Sample Flashcard


---
**FC1** [Chuyển động thẳng đều - Quãng đường]

| | |
|---|---|
| **Front** | Một ô tô khởi hành từ A lúc 7 giờ sáng, chuyển động thẳng đều với vận tốc 60 km/h. Hỏi lúc 9 giờ sáng cùng ngày, ô tô cách A bao nhiêu km? |
| **Back** | 120 km. Thời gian ô tô đã đi là: t = 9 giờ - 7 giờ = 2 giờ. Quãng đường ô tô đi được là: s = v * t = 60 km/h * 2 h = 120 km. |


### Sample Fill-blank


---
**FB1** [Chuyển động thẳng đều - Quãng đường]

> Một ô tô khởi hành từ A lúc 7 giờ sáng với vận tốc 60 km/h. Đến 9 giờ sáng, ô tô đã đi được quãng đường là ___ km.

Answers: `120`


### ✅ All game types test PASSED

---
## Final Summary

In [13]:
# Build summary table
rows = []
total_items = 0
total_time = 0.0
pass_rates = []

for name, s in ALL_RESULTS.items():
    if s is None:
        rows.append(f'| {name} | ❌ FAILED | | | | | |')
        continue
    pr = s['passed'] / s['generated'] * 100 if s['generated'] > 0 else 0
    pass_rates.append(pr)
    total_items += s['total']
    total_time += s['elapsed']
    rows.append(f'| {name} | {s["total"]} | {s["quiz"]} | {s["fc"]} | {s["fb"]} | {pr:.0f}% | {s["elapsed"]:.0f}s |')

avg_pr = sum(pass_rates) / len(pass_rates) if pass_rates else 0
rows.append(f'| **TOTAL** | **{total_items}** | | | | **{avg_pr:.0f}%** | **{total_time:.0f}s** |')

table = '\n'.join(rows)

failed = [k for k, v in ALL_RESULTS.items() if v is None]
status = '❌ Failed: ' + ', '.join(failed) if failed else '✅ All tests passed'

display(Markdown(f"""
### Results

| Test | Items | Quiz | FC | FB | Pass Rate | Time |
|------|-------|------|----|----|-----------|------|
{table}

**Tests:** {len(ALL_RESULTS)} | **Items:** {total_items} | **Time:** {total_time:.0f}s ({total_time/60:.1f} min)

### {status}
"""))


### Results

| Test | Items | Quiz | FC | FB | Pass Rate | Time |
|------|-------|------|----|----|-----------|------|
| accuracy_10q | 10 | 10 | 0 | 0 | 100% | 113s |
| vietnamese | 20 | 10 | 10 | 0 | 62% | 199s |
| diversity_20q | 19 | 19 | 0 | 0 | 95% | 143s |
| scale_30q | 24 | 24 | 0 | 0 | 100% | 245s |
| all_types | 57 | 19 | 19 | 19 | 95% | 223s |
| **TOTAL** | **130** | | | | **90%** | **923s** |

**Tests:** 5 | **Items:** 130 | **Time:** 923s (15.4 min)

### ✅ All tests passed
